# Which model should answer?

*Price a cheap model and a strong one on the same judged questions, then make a router earn its keep against a coin flip.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/flows/04_which_model_should_answer.ipynb)
[![Open in GitHub Codespaces](https://img.shields.io/badge/Open%20in%20Codespaces-2f363d?logo=github&logoColor=white)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Claim.** _To be written from the first full run._
**Result.** _To be written from the first full run._
**Requires.** couchbase · decision-model · local-embeddings
**Read** ~15 min · **Run** ~15 min · **Cost** a few dollars, nearly all of it the strong model reading 160 sets of contract excerpts

Tokens keep getting cheaper. At a fixed level of capability, the price of a token has fallen
by roughly ten times a year ([a16z](https://a16z.com/llmflation-llm-inference-cost/),
[Epoch AI](https://epoch.ai/data-insights/llm-inference-price-trends)). Bills have not fallen
with them. Reasoning models are billed for thousands of output tokens nobody sees. Agents send
their context back on every step: Anthropic measured its multi-agent research system at
[about fifteen times the tokens of a chat](https://www.anthropic.com/engineering/multi-agent-research-system).
And cheaper tokens get spent on work nobody would have paid for before, the
[Jevons paradox](https://fortune.com/2025/01/27/microsoft-ceo-satya-nadella-deepseek-optimism-jevons-paradox)
that Microsoft's chief executive invoked the week reasoning got cheap. The number that matters is
the price per **task**, and the largest single lever on it is which model you send the task to.

The usual answer is a **router**: something that looks at each request and picks a model. It
comes in three shapes, and each bets on something different.

| | decides | using | pays for |
| --- | --- | --- | --- |
| **static assignment** | one model per kind of request | an evaluation, done once | nothing per request |
| **predictive router** | one model per request, before anyone answers | the request | a router call on every request |
| **cascade** | whether to escalate, after the cheap model answers | the cheap answer | the cheap answer on every request, and the strong one too when it escalates |

Routers have a better reputation than their measurements. The well-known 2024 result,
[RouteLLM](https://www.lmsys.org/blog/2024-07-01-routellm/)'s "over 85% cheaper at 95% of
GPT-4's quality", holds on one benchmark and falls to 35–45% on two others. Independent router
benchmarks since then
([LLMRouterBench](https://arxiv.org/html/2601.07206v1),
[the routing plateau](https://arxiv.org/html/2606.07587v1)) find most methods converging on the
same ceiling, far below a perfect router, with several commercial routers failing to beat simply
using the best single model. And [a small 2026 study](https://arxiv.org/abs/2608.23023) found a
fixed table, one model per *type* of question, beating the best single model at under half the
cost.

So this notebook measures a router the way it has to be measured. The application is the
contract question-answering pipeline from [`flows/01`](01_rag_that_you_can_trust.ipynb) and
[`flows/03`](03_when_to_stop_searching.ipynb), in the form `flows/03` found most accurate:
retrieve twenty chunks of one contract from Couchbase and answer once, quoting verbatim. Only the
answering model changes. Both models answer all 160 judged questions, once, and every answer is
graded against a lawyer's annotation with no judge involved. Then each routing strategy is
*replayed* over those answers: which model's answer would it have used, and what would that
have cost? That makes every comparison exact rather than sampled, and it allows two reference
points that any router has to beat:

- **A coin flip at the same price.** Send a random share of the questions to the strong model.
  Its expected accuracy is a straight line between the two models. A router that lands on that
  line has learned nothing about the questions.
- **An oracle.** The cheap model wherever it was right, and the strong one only where the strong
  one alone was right. Nothing does better. The distance between the line and the oracle is
  everything a router could possibly win.

The router is a decision model, TypeSafe's Jev, introduced in
[`experiments/01`](../experiments/01_first_look_at_decision_models.ipynb): it answers typed
questions with probabilities in a few hundred milliseconds, for a fraction of a cent. It
plays both roles here. Before anyone answers, it says how hard a question looks. After the cheap
model answers, it says whether the answer holds up.

**Setup.** A Couchbase cluster, and an [OpenRouter API key](https://openrouter.ai/settings/keys)
in `OPENROUTER_API_KEY` with access to `typesafe/jev`. Both answering models are reached through
OpenRouter too, whichever provider your other notebooks use, so that one price list covers
every call. If you have not run `flows/01`, its index is built here.

In [ ]:
# Setup: load the helpers, install anything missing, check what this notebook needs.
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it. Once: a
        # restarted session keeps the files but not sys.path.
        root = pathlib.Path("cbnb-repo").resolve()
        if not root.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, str(root)])
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["couchbase", "decision-model", "local-embeddings"],
                          extras=["plots"])

## 1. Two models, two prices

A cheap model and a strong one, chosen the way most applications choose: a small and a large
model from one family. The prices are read live from OpenRouter's public model list, so the
arithmetic in this notebook stays current when the price sheet changes. Change the two ids to
compare any pair OpenRouter serves.

In [ ]:
# Choose the two answering models, and read what each charges from OpenRouter's price list.
import pandas as pd

from cbnb.pricing import openrouter_prices

CHEAP, STRONG = "openai/gpt-5-nano", "openai/gpt-5"

prices = openrouter_prices([CHEAP, STRONG])
for role, model in (("cheap", CHEAP), ("strong", STRONG)):
    per_in, per_out = prices[model].per_million
    print(f"{role:6}  {model:24}  ${per_in:6.2f} per million tokens in,  ${per_out:6.2f} out")
print(f"\nThe strong model charges {prices[STRONG].prompt / prices[CHEAP].prompt:.0f}x as much "
      f"per token read, and {prices[STRONG].completion / prices[CHEAP].completion:.0f}x per token "
      f"written.")

A price per token is not a price per answer. How many tokens a model reads is set by the
application: here, the twenty retrieved chunks. How many it writes is partly up to the model,
and a reasoning model writes a great many that never appear in the reply. Every call below is
metered, and priced with one formula:

> **dollars = tokens read × price in + tokens written × price out**

Reasoning tokens count as tokens written and are billed at the output price. The prices are list
prices: no prompt-cache discount, no batch discount.

The contracts are the twenty `flows/01` indexed, in the same overlapping 1,200-character chunks.

In [ ]:
# Load the contracts and split them into overlapping chunks, as flows/01 did.
import pandas as pd

from cbnb.datasets import load_cuad

CHUNK_CHARS, CHUNK_OVERLAP = 1200, 200

cuad = load_cuad()
questions = cuad.questions_with_absences()


def chunk(text, size=CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    """Fixed-width overlapping windows, each carrying its character range."""
    step = size - overlap
    return [
        {"start": start, "end": min(start + size, len(text)), "text": text[start:start + size]}
        for start in range(0, max(len(text) - overlap, 1), step)
    ]


chunks = pd.DataFrame([
    {"chunk_id": f"{row.contract_id}-{number}", "contract_id": row.contract_id,
     "contract_type": row.contract_type, **piece}
    for row in cuad.contracts.itertuples(index=False)
    for number, piece in enumerate(chunk(row.text))
])

# The ground truth: which chunks overlap a span a lawyer marked.
gold = {}
for span in cuad.spans.itertuples(index=False):
    same = chunks[chunks.contract_id == span.contract_id]
    hits = same[(same.start < span.end) & (same.end > span.start)]
    gold.setdefault((span.contract_id, span.category), set()).update(hits.chunk_id)

print(f"{len(chunks):,} chunks from {len(cuad.contracts)} contracts, "
      f"{len(questions)} question/contract pairs "
      f"({(~questions.answerable).mean():.0%} with no answer)")

In [ ]:
# Embed the chunks and index them in Couchbase. Idempotent: flows/01 built this.
from cbnb.couchbase_io import (connect, ensure_collection, ensure_vector_index, upsert_docs,
                               wait_for_index)
from cbnb.embeddings import Embedder

BUCKET, SCOPE, COLLECTION, INDEX = settings.cb_bucket, "contracts", "chunks", "contract_chunks"

embedder = Embedder()
cluster = connect(settings)
collection = ensure_collection(cluster, BUCKET, SCOPE, COLLECTION)

vectors = embedder.encode(chunks.text.tolist())
upsert_docs(collection, {
    row.chunk_id: {
        "type": COLLECTION, "chunk_id": row.chunk_id, "contract_id": int(row.contract_id),
        "contract_key": f"c{row.contract_id}", "contract_type": row.contract_type,
        "start": int(row.start), "end": int(row.end), "text": row.text,
        "embedding": vector.tolist(),
    }
    for row, vector in zip(chunks.itertuples(index=False), vectors)
})
ensure_vector_index(
    cluster, bucket_name=BUCKET, scope_name=SCOPE, collection_name=COLLECTION,
    index_name=INDEX, vector_field="embedding", dims=embedder.dims,
    text_fields=["text"], keyword_fields=["contract_key", "contract_type"],
)
wait_for_index(cluster, bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX,
               expected=len(chunks))

The pipeline is `flows/03`'s single shot at twenty chunks, with one field added to the answer: the
model's own estimate that it is right. That costs nothing extra to ask for, and it is the first
signal anyone reaches for when deciding whether to escalate.

In [ ]:
# Retrieve from one contract in Couchbase, then answer with a chosen model, metering the call.
import couchbase.search as search
from pydantic import BaseModel, Field

from cbnb.couchbase_io import vector_search
from cbnb.llm import LLM, Usage

K = 20
llms = {model: LLM("openrouter", model=model) for model in (CHEAP, STRONG)}
lookup = cuad.categories.set_index("category").question


class Answer(BaseModel):
    found: bool = Field(description="true only if the contract text below answers the question")
    quote: str = Field(description="the exact sentence from the text that supports the answer, copied verbatim; empty if not found")
    answer: str = Field(description="a one-sentence answer, or why the contract does not say")
    confidence: float = Field(description="the probability, from 0 to 1, that this answer is right, including an answer saying the contract does not address the question")


SYSTEM = """You answer questions about a contract using only the excerpts provided.
Quote verbatim from the excerpts. If they do not answer the question, say so."""


def excerpts_for(contract_id, question):
    """The K chunks of one contract nearest the question, joined for a prompt."""
    hits = vector_search(
        cluster, bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX,
        vector_field="embedding", query_vector=embedder.encode_one(question),
        k=K, num_candidates=100, fields=["text"],
        prefilter=search.TermQuery(f"c{contract_id}", field="contract_key"),
    )
    return "\n\n---\n\n".join(hit["text"] for hit in hits)


def answer(model, question, excerpts):
    """One answer, the tokens it took, and what it cost at list price."""
    meter = Usage()
    reply = llms[model].structured(
        f"Contract excerpts:\n\n{excerpts}\n\nQuestion: {question}",
        Answer, system=SYSTEM, max_tokens=2000, meter=meter,
    )
    price = prices[model]
    dollars = meter.prompt_tokens * price.prompt + meter.completion_tokens * price.completion
    return reply, meter, dollars

## 2. The same answer at two prices

Start with one question that every commercial contract answers in a single plain sentence:
which state's law governs it. Both models read the same twenty excerpts.

In [ ]:
# Ask one easy question with both models, and price each answer.
EXAMPLE = (14, "Governing Law")
question = lookup[EXAMPLE[1]]
excerpts = excerpts_for(EXAMPLE[0], question)

print(f"contract c{EXAMPLE[0]}: {question}\n")
spent = {}
for model in (CHEAP, STRONG):
    reply, meter, spent[model] = answer(model, question, excerpts)
    print(model)
    print(f"  quote:   {reply.quote[:110]}")
    print(f"  tokens:  {meter.prompt_tokens:,} read, {meter.completion_tokens:,} written "
          f"({meter.reasoning_tokens:,} of them reasoning)")
    print(f"  cost:    ${spent[model]:.5f}\n")
print(f"The strong model's answer cost {spent[STRONG] / spent[CHEAP]:.0f}x as much.")

Compare the two quotes before the two prices. When they name the same clause, the extra spend
bought nothing on this question. And because a choice made once applies to every request, an
application that sends everything to the strong model pays that multiple on every easy
question it gets.

Then look at the token lines. Both models read the same number of tokens, because the
application decided what they read. The difference in what they *wrote* is the model's own,
and much of it is reasoning the reply never shows.

None of this says the cheap model is good enough. That takes every question, not one.

## 3. Every question, both models

The same 160 questions `flows/01` and `flows/03` used, graded the same deterministic way. The
model's quote either lands inside a span a lawyer marked, or it does not:

- **grounded**: answered, and the quote is inside a marked span. *Correct.*
- **abstained**: said "not in this contract", and it genuinely is not. *Correct.*
- **wrong span**: answered, but quoted something the lawyer did not mark.
- **missed**: said "not in this contract", but it is.
- **hallucinated**: answered a question that has no answer.

Each question is also given a **task type**, the coarse label a static table would route on.
Nine of CUAD's 41 categories ask for a *fact* every contract states somewhere: the parties, the
dates, the governing law. The rest ask whether a kind of *clause* exists at all, and 62% of the
time it does not.

In [ ]:
# Grade an answer against the lawyers' spans, and draw the 160 questions flows/01 used.
CORRECT = {"grounded", "abstained"}
FACTS = {"Document Name", "Parties", "Agreement Date", "Effective Date", "Expiration Date",
         "Renewal Term", "Notice Period To Terminate Renewal", "Governing Law",
         "Warranty Duration"}


def grade(row, found, quote):
    spans = cuad.spans[(cuad.spans.contract_id == row.contract_id)
                       & (cuad.spans.category == row.category)]
    if not row.answerable:
        return "abstained" if not found else "hallucinated"
    if not found:
        return "missed"
    said = " ".join(quote.split())
    for span in spans.itertuples(index=False):
        marked = " ".join(span.text.split())
        if said and (said in marked or marked in said):
            return "grounded"
    return "wrong span"


N_EVAL = 160
sample = questions.sample(N_EVAL, random_state=1729).reset_index(drop=True)
sample["task"] = ["fact" if category in FACTS else "clause" for category in sample.category]
print(f"{len(sample)} questions: {sample.answerable.sum()} answerable, "
      f"{(~sample.answerable).sum()} with no answer")
print(f"{(sample.task == 'fact').sum()} ask for a fact, {(sample.task == 'clause').sum()} "
      f"ask whether a clause exists")

In [ ]:
# Answer every question with both models. Each question is retrieved once and shared.
from concurrent.futures import ThreadPoolExecutor


def both(row):
    question = lookup[row.category]
    excerpts = excerpts_for(row.contract_id, question)
    out = {"contract_id": row.contract_id, "category": row.category, "task": row.task,
           "answerable": bool(row.answerable), "question": question, "excerpts": excerpts}
    for tier, model in (("cheap", CHEAP), ("strong", STRONG)):
        reply, meter, dollars = answer(model, question, excerpts)
        outcome = grade(row, reply.found, reply.quote)
        # Some models state a confidence as a percentage; read 85 as 0.85.
        confidence = reply.confidence / 100 if reply.confidence > 1 else reply.confidence
        out |= {f"{tier}_outcome": outcome, f"{tier}_ok": outcome in CORRECT,
                f"{tier}_found": reply.found, f"{tier}_quote": reply.quote,
                f"{tier}_answer": reply.answer, f"{tier}_confidence": confidence,
                f"{tier}_cost": dollars, f"{tier}_in": meter.prompt_tokens,
                f"{tier}_out": meter.completion_tokens,
                f"{tier}_reasoning": meter.reasoning_tokens, f"{tier}_seconds": meter.seconds}
    return out


with ThreadPoolExecutor(max_workers=8) as pool:
    runs = pd.DataFrame(list(pool.map(both, sample.itertuples(index=False))))
print(f"{len(runs)} questions, each answered by both models")

In [ ]:
# Score and price both models: outcomes, accuracy, dollars per thousand questions, time.
OUTCOMES = ["grounded", "abstained", "wrong span", "missed", "hallucinated"]


def summarise(tier, name):
    counts = runs[f"{tier}_outcome"].value_counts()
    return pd.Series({
        **{outcome: counts.get(outcome, 0) for outcome in OUTCOMES},
        "correct %": 100 * runs[f"{tier}_ok"].mean(),
        "$ per 1,000 questions": 1000 * runs[f"{tier}_cost"].mean(),
        "median seconds": runs[f"{tier}_seconds"].median(),
    }, name=name)


SHOWN = {outcome: "{:,.0f}" for outcome in OUTCOMES}
SHOWN |= {"correct %": "{:.1f}", "$ per 1,000 questions": "${:,.2f}", "median seconds": "{:.1f}"}
pd.DataFrame([summarise("cheap", f"cheap: {CHEAP}"),
              summarise("strong", f"strong: {STRONG}")]).style.format(SHOWN)

That table is the decision most applications make, and make once: one row or the other, for
every request. Read the price column against the accuracy column, and remember that a percentage
on 160 questions moves by a few points on its own.

The questions where the two models *agree* carry no information about routing. Whichever model
answers, the outcome is the same, so the only thing a router can do there is pick the cheaper
one. Everything a router can win or lose is in the disagreements.

In [ ]:
# Count the questions where the two models disagree, and test whether the strong one wins them.
from math import comb


def sign_test(fixed, broke):
    """Exact two-sided sign test: how surprising is this split of the disagreements?"""
    n = fixed + broke
    smaller = min(fixed, broke)
    return min(1.0, 2 * sum(comb(n, i) for i in range(smaller + 1)) / 2 ** n) if n else 1.0


only_strong = int((~runs.cheap_ok & runs.strong_ok).sum())
only_cheap = int((runs.cheap_ok & ~runs.strong_ok).sum())
print(f"strong right, cheap wrong: {only_strong}")
print(f"cheap right, strong wrong: {only_cheap}")
print(f"sign test on those {only_strong + only_cheap}: p = {sign_test(only_strong, only_cheap):.3f}\n")
pd.crosstab(runs.cheap_ok.map({True: "cheap right", False: "cheap wrong"}),
            runs.strong_ok.map({True: "strong right", False: "strong wrong"}),
            rownames=[""], colnames=[""])

Of the four cells, one is the router's whole opportunity: **cheap wrong, strong right**. Those are
the only questions where paying for the strong model changes the outcome for the better. In
the opposite corner, **cheap right, strong wrong**, the expensive model makes things worse, and
any router that sends those questions upward pays more to lose them. On the diagonal the router
can only save money or waste it.

So a router is a bet that it can find the first corner without being told the answers, and
that the first corner is bigger than the second.

Before any router, the coarsest version of that bet: is one task type where the strong model
earns its price?

In [ ]:
# Split by task type: does the strong model earn its price on one kind of question?
by_task = runs.groupby("task").agg(
    questions=("task", "size"),
    cheap_correct=("cheap_ok", "mean"), strong_correct=("strong_ok", "mean"),
    cheap_cost=("cheap_cost", "mean"), strong_cost=("strong_cost", "mean"),
)
by_task[["cheap_correct", "strong_correct"]] *= 100
by_task[["cheap_cost", "strong_cost"]] *= 1000
by_task.columns = ["questions", "cheap correct %", "strong correct %",
                   "cheap $ per 1,000", "strong $ per 1,000"]
by_task.style.format({"questions": "{:.0f}", "cheap correct %": "{:.1f}",
                      "strong correct %": "{:.1f}", "cheap $ per 1,000": "${:,.2f}",
                      "strong $ per 1,000": "${:,.2f}"})

## 4. The lines a router has to beat

Every strategy from here on is a rule for which questions get the strong model's answer. Because
both answers already exist, each rule can be replayed exactly: its accuracy is the share of
questions whose chosen answer was right, and its cost is what the chosen calls cost, plus
anything the rule itself spent deciding.

Two kinds of strategy pay differently, and the replay keeps them apart. A **router** decides
before anyone answers, so a question it sends upward costs only the strong call. A **cascade**
waits for the cheap answer and then decides, so it pays for the cheap call on every question,
including the ones it escalates.

In [ ]:
# Replay a routing rule over the stored answers: its accuracy, its cost, and its latency.
import numpy as np

n = len(runs)
cheap_ok, strong_ok = runs.cheap_ok.to_numpy(), runs.strong_ok.to_numpy()
cheap_cost, strong_cost = runs.cheap_cost.to_numpy(), runs.strong_cost.to_numpy()
cheap_s, strong_s = runs.cheap_seconds.to_numpy(), runs.strong_seconds.to_numpy()


def point(name, to_strong, overhead=0.0, cascade=False, wait=0.0):
    """What a rule does when `to_strong` marks the questions given the strong model's answer.

    `overhead` is dollars spent per question on deciding, and `wait` the seconds it took.
    A cascade has already paid for, and waited for, the cheap answer on every question.
    """
    to_strong = np.asarray(to_strong, bool)
    overhead, wait = np.asarray(overhead, float), np.asarray(wait, float)
    kept = 1.0 if cascade else ~to_strong
    dollars = np.where(to_strong, strong_cost, 0.0) + kept * cheap_cost + overhead
    seconds = np.where(to_strong, strong_s, 0.0) + kept * cheap_s + wait
    chosen_ok = np.where(to_strong, strong_ok, cheap_ok)
    return {"strategy": name, "to strong %": 100 * to_strong.mean(),
            "correct %": 100 * chosen_ok.mean(),
            "$ per 1,000": 1000 * dollars.mean(),
            "p50 seconds": np.percentile(seconds, 50), "p95 seconds": np.percentile(seconds, 95),
            # Against paying for the strong model on everything: questions won and lost.
            "vs all strong: fixed": int((chosen_ok & ~strong_ok).sum()),
            "vs all strong: broke": int((~chosen_ok & strong_ok).sum())}


def coin_flip(dollars_per_1000):
    """Expected accuracy of sending a random share of questions upward, at the same cost."""
    low, high = 1000 * cheap_cost.mean(), 1000 * strong_cost.mean()
    share = (dollars_per_1000 - low) / (high - low)
    return 100 * (cheap_ok.mean() + share * (strong_ok.mean() - cheap_ok.mean()))


oracle = ~cheap_ok & strong_ok
references = pd.DataFrame([
    point("all cheap", np.zeros(n, bool)),
    point("all strong", np.ones(n, bool)),
    point("oracle", oracle),
    point("static: clauses to strong", runs.task.eq("clause")),
    point("static: facts to strong", runs.task.eq("fact")),
]).set_index("strategy")
references["coin flip at that cost"] = references["$ per 1,000"].map(coin_flip)

BOARD = {"to strong %": "{:.0f}", "correct %": "{:.1f}", "$ per 1,000": "${:,.2f}",
         "p50 seconds": "{:.1f}", "p95 seconds": "{:.1f}", "coin flip at that cost": "{:.1f}",
         "vs all strong: fixed": "{:.0f}", "vs all strong: broke": "{:.0f}"}
references.style.format(BOARD)

Read **oracle** first. It is the best any rule could do with these two models' answers. By
construction it is at least as accurate as either model alone, and it sends upward only the
questions in the *cheap wrong, strong right* corner. That is the number router pitches are built
on. A rule that knew the answers in advance could reach it, and no real rule does.

The two **static** rows are the cheapest real alternative: route by task type, decided once, with
no model in the loop. Compare each one's accuracy with the coin flip at its cost. A static row
above its coin flip is a table that knows something about the questions.

## 5. A router that reads the question first

The predictive router asks the decision model two things before anyone answers: how hard the
question looks, and whether the excerpts plainly state an answer. It sees exactly what the
answering model will see, the question and the twenty excerpts, and none of the answers.

In [ ]:
# Before anyone answers, ask the decision model how hard each question looks.
from cbnb.decisions import decide, noul, score

ROUTER = {
    "difficulty": score("How hard is it to answer this question correctly from these contract excerpts?", [
        "Easy: the answer is stated plainly, or plainly absent",
        "Moderate: it takes careful reading",
        "Hard: it takes legal judgement, or the excerpts are ambiguous",
    ]),
    "stated": noul("Do these excerpts contain a sentence that directly answers the question?"),
}


def state(row):
    return f"Question: {row.question}\n\nContract excerpts:\n\n{row.excerpts}"


with ThreadPoolExecutor(max_workers=8) as pool:
    routed = list(pool.map(lambda row: decide(state(row), ROUTER), runs.itertuples(index=False)))

runs["difficulty"] = [d.value("difficulty") for d in routed]
runs["stated"] = [d.value("stated") for d in routed]
runs["router_cost"] = [d.cost for d in routed]
runs["router_seconds"] = [d.seconds for d in routed]
print(f"{len(routed)} decisions: ${runs.router_cost.sum():.4f} in all, "
      f"median {1000 * runs.router_seconds.median():.0f} ms each")

Before building anything on those numbers, check whether they know anything. A routing signal has
one job: rank the questions the cheap model gets wrong above the ones it gets right. **AUC**
measures that directly. It is the chance that a question the cheap model failed looks riskier
than one it passed: 0.5 is a coin, 1.0 is perfect.

The second column is stricter, and it is the one that matters for routing. It asks whether the
signal finds the questions where **the strong model would fix** the cheap one's mistake. A
question both models get wrong is a mistake no router can repair, and sending it upward only
costs money.

In [ ]:
# How well does each signal rank the cheap model's mistakes, and the ones the strong model fixes?
def auc(risk, positive):
    """The chance a positive question looks riskier than a negative one. 0.5 is a coin."""
    risk, positive = np.asarray(risk, float), np.asarray(positive, bool)
    hit, miss = risk[positive], risk[~positive]
    if not len(hit) or not len(miss):
        return float("nan")
    return float(np.mean([(h > m) + 0.5 * (h == m) for h in hit for m in miss]))


def signal_row(name, risk):
    return pd.Series({"AUC: cheap is wrong": auc(risk, ~cheap_ok),
                      "AUC: strong would fix it": auc(risk, oracle)}, name=name)


signals = pd.DataFrame([
    signal_row("router: difficulty", runs.difficulty),
    signal_row("router: no answer stated", 1 - runs.stated),
])
signals.style.format("{:.2f}")

Now turn the better signal into a router and sweep its threshold. At one extreme every question
stays with the cheap model; at the other every question goes up. In between, the questions the
router considers riskiest go up first. Each step of the sweep is a router someone could deploy,
and every one of them pays for a decision model call on every question.

In [ ]:
# Sweep a rule's threshold: send the riskiest questions upward, one more at a time.
def sweep(name, risk, overhead=0.0, cascade=False, wait=0.0):
    order = np.argsort(-np.asarray(risk, float), kind="stable")
    rows = []
    for m in range(n + 1):
        to_strong = np.zeros(n, bool)
        to_strong[order[:m]] = True
        rows.append(point(name, to_strong, overhead, cascade, wait))
    return pd.DataFrame(rows)


curves = {"router: difficulty": sweep("router: difficulty", runs.difficulty,
                                      runs.router_cost.to_numpy(), wait=runs.router_seconds.to_numpy())}

In [ ]:
# Draw the strategies on one plane: dollars per thousand questions against accuracy.
import matplotlib.pyplot as plt

COLORS = {"router: difficulty": "#2a78d6", "cascade: own confidence": "#eb6834",
          "cascade: decision model check": "#1baf7a"}
INK, MUTED = "#0b0b0b", "#8a8985"


def chart(curves, title):
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ends = references.loc[["all cheap", "all strong"]]
    ax.plot(ends["$ per 1,000"], ends["correct %"], color=MUTED, linestyle="--", linewidth=1.5,
            label="coin flip between the two models")
    for name, curve in curves.items():
        ax.plot(curve["$ per 1,000"], curve["correct %"], color=COLORS[name], linewidth=2,
                label=name)
    for name, marker in (("all cheap", "o"), ("all strong", "o"), ("oracle", "*"),
                         ("static: clauses to strong", "s"), ("static: facts to strong", "s")):
        row = references.loc[name]
        ax.scatter(row["$ per 1,000"], row["correct %"], marker=marker, color=INK,
                   s=140 if marker == "*" else 40, zorder=3, edgecolors="white", linewidths=1.5)
        ax.annotate(name, (row["$ per 1,000"], row["correct %"]), textcoords="offset points",
                    xytext=(6, -12), fontsize=8, color=INK)
    ax.set_xlabel("dollars per 1,000 questions, at list price")
    ax.set_ylabel("correct %")
    ax.set_title(title, loc="left")
    ax.legend(frameon=False, fontsize=8, ncols=2, loc="upper center", bbox_to_anchor=(0.5, -0.14))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


chart(curves, f"A predictive router against its references (n={n})")

How to read it. The dashed line is the coin flip: what you get at each price by sending that
share of questions upward at random. The star is the oracle. A curve that rides above the
dashed line has learned which questions need the strong model; a curve that hugs it has learned
nothing; a curve below it is sending the wrong questions upward. The gap between the dashed line
and the star is all there was to win.

Notice also where the curve starts. With nothing sent upward it sits to the right of the cheap
model's dot by the cost of the decision calls, and above it by nothing. A router's own price is
paid on every question whether it helps or not.

## 6. Answer cheaply, check, escalate

A cascade has information a router does not: the cheap model's actual answer. Two ways to judge it:

- **The model's own confidence.** Already in every answer, so free. But models trained to be
  helpful are known to be overconfident about their own answers:
  [GPT-4's calibration got worse after RLHF](https://arxiv.org/pdf/2303.08774v4).
- **A check by the decision model.** Show it the question, the excerpts and the cheap answer, and
  ask whether the answer holds up. One more call per question, at the decision model's price.

In [ ]:
# After the cheap model answers, ask the decision model whether its answer holds up.
CHECK = {"holds": noul(
    "Is the proposed answer right, judged only against these excerpts? An answer saying the "
    "contract does not address the question is right only if no excerpt addresses it.")}


def checked_state(row):
    quote = row.cheap_quote if row.cheap_found else "(none: it says the contract does not address this)"
    return f"{state(row)}\n\nProposed answer: {row.cheap_answer}\nSupporting quote: {quote}"


with ThreadPoolExecutor(max_workers=8) as pool:
    checks = list(pool.map(lambda row: decide(checked_state(row), CHECK),
                           runs.itertuples(index=False)))

runs["holds"] = [d.value("holds") for d in checks]
runs["check_cost"] = [d.cost for d in checks]
runs["check_seconds"] = [d.seconds for d in checks]

signals = pd.concat([signals, pd.DataFrame([
    signal_row("cascade: own confidence", 1 - runs.cheap_confidence),
    signal_row("cascade: decision model check", 1 - runs.holds),
])])
signals.style.format("{:.2f}")

The cascade signals see the answer, so they ought to rank the cheap model's mistakes better than
anything that only saw the question. Whether they do is in the table. Look at the second column
before the first again: a check that is good at spotting wrong answers but cannot tell which ones
the strong model would fix escalates questions that come back just as wrong.

In [ ]:
# Add both cascades to the picture.
curves["cascade: own confidence"] = sweep(
    "cascade: own confidence", 1 - runs.cheap_confidence, cascade=True)
curves["cascade: decision model check"] = sweep(
    "cascade: decision model check", 1 - runs.holds, runs.check_cost.to_numpy(), cascade=True,
    wait=runs.check_seconds.to_numpy())
chart(curves, f"Routers and cascades against their references (n={n})")

A cascade's curve starts at the cheap model's accuracy and pays the cheap model's price
everywhere along it. When it escalates everything, it is more expensive than simply using the
strong model, because it paid for the cheap answer first. So a cascade has to win early: its
whole case rests on the first few escalations being the right ones.

## 7. Pick a threshold, then test it on questions it did not see

A curve is a menu of thresholds, and choosing one by looking at the curve is choosing with the
answers in hand. A deployed router gets one threshold, picked in advance, and meets questions it
has not seen. So the scoreboard below is honest about that: each rule's threshold is picked on
half of the questions and scored on the other half, then the halves swap, so every question is
scored once by a threshold that never saw it.

The budget is the same for every rule: **half of what the strong model costs per question.**
Each rule escalates as many questions as that budget allows on the half it learns from.

In [ ]:
# Choose each rule's threshold on half the questions, then score it on the other half.
BUDGET = strong_cost.mean() / 2   # dollars per question


def held_out(name, risk, overhead=0.0, cascade=False, wait=0.0):
    risk = np.asarray(risk, float)
    overhead = np.broadcast_to(np.asarray(overhead, float), (n,))
    halves = np.arange(n) % 2
    to_strong = np.zeros(n, bool)
    for fold in (0, 1):
        learn, test = halves != fold, halves == fold
        kept_cost = 0.0 if cascade else cheap_cost[learn]
        base = overhead[learn] + (cheap_cost[learn] if cascade else 0.0)
        # Lower the threshold one distinct value at a time, so tied scores move together.
        threshold = np.inf
        for candidate in np.unique(risk[learn])[::-1]:
            sent = risk[learn] >= candidate
            if (base + np.where(sent, strong_cost[learn], kept_cost)).mean() > BUDGET:
                break
            threshold = candidate
        to_strong[test] = risk[test] >= threshold
    return point(name, to_strong, overhead, cascade, wait)


rules = pd.DataFrame([
    held_out("router: difficulty", runs.difficulty, runs.router_cost, wait=runs.router_seconds),
    held_out("cascade: own confidence", 1 - runs.cheap_confidence, cascade=True),
    held_out("cascade: decision model check", 1 - runs.holds, runs.check_cost, cascade=True,
             wait=runs.check_seconds),
]).set_index("strategy")

In [ ]:
# The scoreboard: every rule against the coin flip at its own cost, and against the oracle.
board = pd.concat([references, rules])
board["coin flip at that cost"] = board["$ per 1,000"].map(coin_flip)
board["gain over coin flip"] = board["correct %"] - board["coin flip at that cost"]
board["share of the oracle's gain %"] = (
    100 * board["gain over coin flip"] / board.loc["oracle", "gain over coin flip"])
board["p vs all strong"] = [sign_test(fixed, broke) for fixed, broke
                            in zip(board["vs all strong: fixed"], board["vs all strong: broke"])]

COLUMNS = ["to strong %", "correct %", "coin flip at that cost", "gain over coin flip",
           "share of the oracle's gain %", "$ per 1,000", "p50 seconds", "p95 seconds",
           "vs all strong: fixed", "vs all strong: broke", "p vs all strong"]
board[COLUMNS].style.format(BOARD | {
    "gain over coin flip": "{:+.1f}", "share of the oracle's gain %": "{:.0f}",
    "p vs all strong": "{:.3f}"})

The columns, in the order they matter:

- **gain over coin flip** is accuracy above what a random split would get *at the same price*.
  It is the only column that credits a rule for knowing something about the questions. A rule
  that spends more simply buys more accuracy, which a coin flip would also buy.
- **share of the oracle's gain** puts that in proportion: how much of what a perfect router
  could win this rule actually won. Router benchmarks published in 2026
  ([one example](https://arxiv.org/html/2608.08265)) put the best deployable routers at
  roughly a tenth.
- **fixed / broke vs all strong**, with its sign test, asks the question a buyer asks: compared
  with paying for the strong model on everything, which questions did this rule get right that
  the strong model missed, and which did it lose?
- **p50 / p95 seconds** are what a user waits. A cascade's slow tail is the cheap call, the check
  and the strong call, one after another.

## 8. Where the money went

Routing changes *which* model reads and writes. It does not change *how much* is read. Split
each bill into its three parts: reading the excerpts, writing the answer, and reasoning the
reply never shows.

In [ ]:
# Split each model's bill into reading, writing, and hidden reasoning.
def anatomy(tier, model):
    price = prices[model]
    read = runs[f"{tier}_in"].sum() * price.prompt
    think = runs[f"{tier}_reasoning"].sum() * price.completion
    write = (runs[f"{tier}_out"].sum() - runs[f"{tier}_reasoning"].sum()) * price.completion
    total = read + think + write
    return pd.Series({
        "tokens read per question": runs[f"{tier}_in"].mean(),
        "tokens written per question": runs[f"{tier}_out"].mean(),
        "of which reasoning": runs[f"{tier}_reasoning"].mean(),
        "reading %": 100 * read / total, "reasoning %": 100 * think / total,
        "writing %": 100 * write / total, "$ for all questions": total,
    }, name=model)


decisions = pd.Series({"reading %": 100.0, "reasoning %": 0.0, "writing %": 0.0,
                       "$ for all questions": runs.router_cost.sum() + runs.check_cost.sum()},
                      name="decision model, both roles")
pd.DataFrame([anatomy("cheap", CHEAP), anatomy("strong", STRONG), decisions]).style.format({
    "tokens read per question": "{:,.0f}", "tokens written per question": "{:,.0f}",
    "of which reasoning": "{:,.0f}", "reading %": "{:.0f}", "reasoning %": "{:.0f}",
    "writing %": "{:.0f}", "$ for all questions": "${:,.4f}"}, na_rep="")

Three things to read off that table.

**What the application chose to send.** Both models read the same twenty chunks per question.
`flows/03` chose twenty because it was more accurate than four, and that choice multiplied the
reading part of every bill here by five, for both models. It is as much a cost decision as
the choice of model, and nobody calls it routing.

**What the model chose to think.** A reasoning model's hidden tokens are billed at the output
price, the expensive side of the sheet. If the reasoning column is large, the strong model's
price per answer was set as much by how long it thought as by its price per token.

**What deciding cost.** The decision model's row is both of its roles, on every question. Set it
against the strong model's: deciding is cheap enough to be a rounding error, which is the
point of a decision model. Whether the decisions were *good* is the scoreboard's question, not
this table's.

## Where to take this

- **Run it again before believing a gap.** A question one model gets right by luck looks routable
  in a single run, which inflates the oracle. Re-run the answering cell with the cache off: a
  question that flips between runs is noise, not an opportunity.
- **Use all 820 questions for a real table.** Two task types is the coarsest table there is. One
  row per CUAD category needs more than four questions per category to be chosen honestly.
- **Treat reasoning effort as a third model.** The same model at low and high effort is a routing
  choice with no second vendor, and every major provider now has the dial.
- **Spend less on reading.** Price accuracy at four, ten and twenty chunks for both models. A cheap
  model reading more may beat a strong one reading less, at a fraction of the price.
- **Keep the decisions.** Store each question's route, scores and graded outcome as a document in
  Couchbase. A threshold is only as good as the traffic it was set on, and this is how it gets
  set again when the traffic or the price list changes.

## What produced these results

Every number above came from the models below. A different model, or the same name served
differently, will give different numbers.

In [ ]:
# Name the models that produced the results above.
cbnb.run_card()

In [ ]:
# Optional clean-up: delete what this notebook created. Uncomment to run.
# It lives in the scope flows/01 created, which this notebook reuses.
# from cbnb.couchbase_io import drop_demo_data
# drop_demo_data(cluster, BUCKET, SCOPE)